# Etapa 6 - Arquitetura-Alvo em Nuvem

Registrar a arquitetura proposta para dados, inferencia, API e observabilidade.

## Arquitetura implantada (AWS)

Decisao: **AWS** como provedor. Nao ficou so no papel - a arquitetura foi implementada em Terraform e implantada de verdade (regiao `us-east-2`), com os endpoints validados ponta a ponta (`/recomendar`, `/feedback`, `/stats`, UI do MLflow). Detalhamento completo, runbook e IaC em `deploy/aws/` (ver tambem `README.md`, secao "Arquitetura-Alvo em Nuvem (AWS)").

- **Compute**: ECR + ECS Fargate (ARM64/Graviton), containers de `deploy/Dockerfile.fastapi` e `deploy/Dockerfile.mlflow`, atras de um Application Load Balancer
- **Object Storage**: S3 para o artifact store do MLflow
- **Estado do bandit**: DynamoDB (substitui o pickle local, permite multiplas replicas) - contadores + log de decisoes
- **MLflow backend store**: RDS PostgreSQL (substitui o SQLite local)
- **Monitoramento**: CloudWatch Logs
- **Segredos**: Secrets Manager (credenciais do RDS)
- **IAM**: usuario de deploy com politica dedicada (`deploy/aws/iam/deploy-user-policy.json`), sem AdministratorAccess

Durante o deploy real apareceram bugs que nao existiam rodando so localmente (ex.: incompatibilidade do uvloop/httptools no Fargate ARM64, Host header do healthcheck da ALB, grace period do ECS) - todos documentados e corrigidos, detalhes em `doc/PLANO_EXECUCAO.md` (Etapa 6) e `deploy/aws/README.md`.


## Validacao pratica: local (Docker Compose) vs AWS

Testa as mesmas rotas da API (`/health`, `/`, `/recomendar`, `/feedback`, `/stats`) e o MLflow contra os dois ambientes, para confirmar que as duas arquiteturas (local e nuvem) estao funcionando de ponta a ponta com o mesmo contrato.

**Pre-requisitos:**
- **Local**: suba o Docker Compose antes de rodar esta celula -
  ```bash
  docker compose -f ../deploy/docker-compose.yml up -d --force-recreate
  ```
- **AWS**: o stack fica pausado (`desired_count=0`) fora de sessoes de dev/demo para nao gerar custo. Se estiver pausado, retome antes de rodar -
  ```bash
  export AWS_PROFILE=datathon AWS_REGION=us-east-2
  aws ecs update-service --cluster datathon-bandit-cluster --service datathon-bandit-fastapi --desired-count 1
  aws ecs update-service --cluster datathon-bandit-cluster --service datathon-bandit-mlflow  --desired-count 1
  ```
  Leva ~1-2 min para os endpoints responderem. Se o DNS do ALB mudou (ex.: `terraform apply` recriou o load balancer), atualize `ENVIRONMENTS["aws"]` abaixo com `terraform -chdir=../deploy/aws/terraform output -raw alb_dns_name`.


In [1]:
import requests

ENVIRONMENTS = {
    "local": {
        "api": "http://localhost:8000",
        "mlflow": "http://localhost:5002",
    },
    "aws": {
        "api": "http://datathon-bandit-alb-361652049.us-east-2.elb.amazonaws.com",
        "mlflow": "http://datathon-bandit-alb-361652049.us-east-2.elb.amazonaws.com:5000",
    },
}


### Funcao de validacao

Roda um smoke test completo contra um ambiente e imprime `✅`/`❌` por rota. Cobre tambem os casos de erro esperados da API (`404` para `decision_id` inexistente, `409` para feedback duplicado), que ja sao tratados em `app/main.py`.


In [2]:
def validar_ambiente(nome: str, urls: dict, timeout: int = 8) -> dict:
    resultado = {"ambiente": nome, "checks": []}

    def registrar(rota, ok, detalhe=""):
        resultado["checks"].append({"rota": rota, "ok": ok, "detalhe": detalhe})
        print(f"{'✅' if ok else '❌'} [{nome}] {rota} {detalhe}")

    # GET /health
    try:
        r = requests.get(f"{urls['api']}/health", timeout=timeout)
        registrar("GET /health", r.status_code == 200, f"status={r.status_code}")
    except Exception as exc:
        registrar("GET /health", False, f"erro: {exc}")
        return resultado  # sem health de pe, nao adianta seguir

    # GET / -> redirect para /docs
    try:
        r = requests.get(f"{urls['api']}/", timeout=timeout, allow_redirects=False)
        registrar("GET / (redirect /docs)", r.status_code in (307, 308), f"status={r.status_code}")
    except Exception as exc:
        registrar("GET /", False, f"erro: {exc}")

    # POST /recomendar
    decision_id = None
    try:
        contexto = {"idade": 42, "poutcome": "unknown", "previous": 1}
        r = requests.post(f"{urls['api']}/recomendar", json=contexto, timeout=timeout)
        ok = r.status_code == 200 and "decision_id" in r.json()
        decision_id = r.json().get("decision_id") if ok else None
        registrar("POST /recomendar", ok, f"resposta={r.json() if ok else r.text}")
    except Exception as exc:
        registrar("POST /recomendar", False, f"erro: {exc}")

    # POST /feedback (caminho feliz)
    if decision_id:
        try:
            r = requests.post(
                f"{urls['api']}/feedback",
                json={"decision_id": decision_id, "converteu": True},
                timeout=timeout,
            )
            registrar("POST /feedback", r.status_code == 200, f"resposta={r.json() if r.status_code == 200 else r.text}")
        except Exception as exc:
            registrar("POST /feedback", False, f"erro: {exc}")

        # Mesmo decision_id de novo -> espera 409 (evita contar o evento 2x)
        try:
            r = requests.post(
                f"{urls['api']}/feedback",
                json={"decision_id": decision_id, "converteu": True},
                timeout=timeout,
            )
            registrar("POST /feedback duplicado (espera 409)", r.status_code == 409, f"status={r.status_code}")
        except Exception as exc:
            registrar("POST /feedback duplicado", False, f"erro: {exc}")
    else:
        registrar("POST /feedback", False, "pulado: sem decision_id valido")

    # decision_id inexistente -> espera 404
    try:
        r = requests.post(
            f"{urls['api']}/feedback",
            json={"decision_id": "00000000-0000-0000-0000-000000000000", "converteu": True},
            timeout=timeout,
        )
        registrar("POST /feedback id inexistente (espera 404)", r.status_code == 404, f"status={r.status_code}")
    except Exception as exc:
        registrar("POST /feedback id inexistente", False, f"erro: {exc}")

    # GET /stats
    try:
        r = requests.get(f"{urls['api']}/stats", timeout=timeout)
        registrar("GET /stats", r.status_code == 200, f"resposta={r.json() if r.status_code == 200 else r.text}")
    except Exception as exc:
        registrar("GET /stats", False, f"erro: {exc}")

    # MLflow (API REST, nao so a UI)
    try:
        r = requests.get(f"{urls['mlflow']}/api/2.0/mlflow/experiments/search?max_results=5", timeout=timeout)
        registrar("GET MLflow experiments/search", r.status_code == 200, f"status={r.status_code}")
    except Exception as exc:
        registrar("GET MLflow experiments/search", False, f"erro: {exc}")

    return resultado


### Rodar contra o ambiente local (Docker Compose)


In [3]:
resultado_local = validar_ambiente("local (docker-compose)", ENVIRONMENTS["local"])


✅ [local (docker-compose)] GET /health status=200
✅ [local (docker-compose)] GET / (redirect /docs) status=307
✅ [local (docker-compose)] POST /recomendar resposta={'decision_id': 'f4f0728c-3123-4b9b-8af9-c7646589a588', 'arm': 'cellular'}
✅ [local (docker-compose)] POST /feedback resposta={'decision_id': 'f4f0728c-3123-4b9b-8af9-c7646589a588', 'arm': 'cellular', 'reward': 1}
✅ [local (docker-compose)] POST /feedback duplicado (espera 409) status=409
✅ [local (docker-compose)] POST /feedback id inexistente (espera 404) status=404
✅ [local (docker-compose)] GET /stats resposta={'braços': {'cellular': {'observações': 29286, 'conversões': 4370, 'taxa_de_conversao_estimada': 0.1541}, 'telephone': {'observações': 2906, 'conversões': 390, 'taxa_de_conversao_estimada': 0.1375}}}


✅ [local (docker-compose)] GET MLflow experiments/search status=200


### Rodar contra o ambiente AWS (ECS Fargate)


In [4]:
resultado_aws = validar_ambiente("AWS (ECS Fargate)", ENVIRONMENTS["aws"])


✅ [AWS (ECS Fargate)] GET /health status=200


✅ [AWS (ECS Fargate)] GET / (redirect /docs) status=307


✅ [AWS (ECS Fargate)] POST /recomendar resposta={'decision_id': 'f279d3fb-74b7-43dd-811d-84924dc58d69', 'arm': 'cellular'}


✅ [AWS (ECS Fargate)] POST /feedback resposta={'decision_id': 'f279d3fb-74b7-43dd-811d-84924dc58d69', 'arm': 'cellular', 'reward': 1}


✅ [AWS (ECS Fargate)] POST /feedback duplicado (espera 409) status=409


✅ [AWS (ECS Fargate)] POST /feedback id inexistente (espera 404) status=404


✅ [AWS (ECS Fargate)] GET /stats resposta={'braços': {'cellular': {'observações': 29307, 'conversões': 4373, 'taxa_de_conversao_estimada': 0.1492}, 'telephone': {'observações': 2906, 'conversões': 390, 'taxa_de_conversao_estimada': 0.1354}}}


✅ [AWS (ECS Fargate)] GET MLflow experiments/search status=200


### Resumo comparativo


In [5]:
import pandas as pd

def resumo(resultado):
    total = len(resultado["checks"])
    ok = sum(1 for c in resultado["checks"] if c["ok"])
    falhas = [c["rota"] for c in resultado["checks"] if not c["ok"]]
    return {
        "ambiente": resultado["ambiente"],
        "checks_ok": f"{ok}/{total}",
        "falhas": ", ".join(falhas) if falhas else "-",
    }

pd.DataFrame([resumo(resultado_local), resumo(resultado_aws)])


,ambiente,checks_ok,falhas
0,local (docker-compose),8/8,-
1,AWS (ECS Fargate),8/8,-
